In [40]:
import pandas as pd
import numpy as np
import pickle
import ast
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

### Load model

In [22]:
# Load Vietnamese SBERT model
model = SentenceTransformer('keepitreal/vietnamese-sbert')
print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")

   Embedding dimension: 768


### Load embeddings

In [41]:
with open("recipes_embeddings_list.pkl", "rb") as f:
    recipes_embeddings_list = pickle.load(f)
print(f"Loaded {len(recipes_embeddings_list)} recipe embeddings")

Loaded 10263 recipe embeddings


### Load dataset

In [24]:
all_recipes_df = pd.read_csv("../../data/all_recipes_final.csv")

print(f"Loaded dataset: {len(all_recipes_df)} recipes")
print(f"Columns: {all_recipes_df.columns.tolist()}")

Loaded dataset: 10263 recipes
Columns: ['title', 'type_of_food', 'link', 'description', 'ingredients', 'step', 'note', 'num_of_ingredients', 'cook_time', 'num_of_people', 'calories', 'source']


### 3. Late Fusion Search Function

**Late Fusion Strategy:**
1. Tính similarity của query với **TẤT CẢ** các câu trong mỗi món
2. Lấy **trung bình** (average) similarity của tất cả câu → điểm số của món
3. Rank tất cả món theo điểm trung bình → Top K

In [25]:
def search_recipes_late_fusion(query, model, recipes_embeddings_list, all_recipes_df, top_k=10):
    """
    Search recipes using LATE FUSION strategy (Average Similarity)

    Late Fusion = Tính similarity với TẤT CẢ câu trong món → Average → Rank

    Args:
        query: User's search query (Vietnamese)
        model: SentenceTransformer model
        recipes_embeddings_list: List of embeddings per dish
        all_recipes_df: Recipe metadata dataframe
        top_k: Number of results to return

    Returns:
        DataFrame with top_k recipes and similarity scores
    """
    # 1. Encode query
    query_embedding = model.encode([query])
    query_embedding = query_embedding / np.linalg.norm(query_embedding)  # Normalize

    # 2. Calculate average similarity for EACH recipe
    recipe_scores = []

    for recipe_idx, dish_embeds in enumerate(recipes_embeddings_list):
        if len(dish_embeds) == 0:
            continue

        # Normalize dish embeddings
        dish_embeds_norm = dish_embeds / np.linalg.norm(dish_embeds, axis=1, keepdims=True)

        # Compute cosine similarity với TẤT CẢ câu
        similarities = np.dot(dish_embeds_norm, query_embedding.T).flatten()

        # LATE FUSION: Average similarity
        avg_similarity = np.mean(similarities)

        recipe_scores.append({
            'recipe_idx': recipe_idx,
            'avg_similarity': float(avg_similarity),
            'max_similarity': float(np.max(similarities)),
            'min_similarity': float(np.min(similarities)),
            'num_sentences': len(similarities)
        })

    # 3. Sort by average similarity
    recipe_scores.sort(key=lambda x: x['avg_similarity'], reverse=True)
    top_recipes = recipe_scores[:top_k]

    # 4. Create results dataframe with FULL recipe info
    results = []
    for item in top_recipes:
        recipe_idx = item['recipe_idx']
        recipe = all_recipes_df.iloc[recipe_idx]

        results.append({
            'recipe_idx': recipe_idx,
            'avg_similarity': item['avg_similarity'],
            'max_similarity': item['max_similarity'],
            'min_similarity': item['min_similarity'],
            'num_sentences': item['num_sentences'],
            'title': recipe['title'],
            'type_of_food': recipe['type_of_food'],
            'cook_time': recipe['cook_time'],
            'num_of_people': recipe['num_of_people'],
            'ingredients': recipe['ingredients'],
            'step': recipe['step'],
            'note': recipe['note'],
            'description': recipe['description'],
            'link': recipe['link']  # Thêm link
        })

    return pd.DataFrame(results)

### 4. Display results function

In [26]:
def parse_list_field(field_value):
    """
    Parse string / list field to Python list safely.
    """
    if pd.isna(field_value):
        return []

    if isinstance(field_value, list):
        return field_value

    if isinstance(field_value, str):
        try:
            parsed = ast.literal_eval(field_value)
            if isinstance(parsed, list):
                return parsed
            return []
        except (ValueError, SyntaxError):
            # fallback: split by comma
            return [s.strip() for s in field_value.split(",") if s.strip()]

    return []

In [27]:
import re

def display_results(results_df, query):
    """
    Display search results với TẤT CẢ thông tin món ăn

    Args:
        results_df: DataFrame from search_recipes_late_fusion
        query: Original query string
    """
    print(f"Query: '{query}'")
    print(f"Top {len(results_df)} Results:")

    for idx, row in results_df.iterrows():
        print(f"\n{'='*100}")
        print(f"{idx+1}. [{row['avg_similarity']:.4f}] {row['title']}")
        print(f"{'='*100}")

        # Basic info
        print(f"\nTHÔNG TIN CƠ BẢN:")
        print(f"   • Loại món: {row['type_of_food']}")
        print(f"   • Thời gian nấu: {row['cook_time']}")
        print(f"   • Số người ăn: {row['num_of_people']}")
        
        # Link
        if pd.notna(row['link']):
            print(f"   • Link: {row['link']}")

        # Similarity scores
        print(f"\nĐIỂM SIMILARITY:")
        print(f"   • Trung bình (AVG): {row['avg_similarity']:.4f}")
        print(f"   • Cao nhất (MAX): {row['max_similarity']:.4f}")
        print(f"   • Thấp nhất (MIN): {row['min_similarity']:.4f}")
        print(f"   • Số câu đánh giá: {row['num_sentences']}")

        # Description
        if pd.notna(row['description']):
            print(f"\nMÔ TẢ:")
            print(f"   {row['description']}")

        # Ingredients
        ingredients = parse_list_field(row['ingredients'])
        if ingredients:
            print(f"\nNGUYÊN LIỆU ({len(ingredients)} món):")
            for i, ing in enumerate(ingredients, 1):
                print(f"   {i}. {ing}")

        # Steps
        steps = parse_list_field(row['step'])
        if steps:
            # 1. Gộp tất cả step thành 1 chuỗi
            steps_text = " ".join(step.strip() for step in steps)

            # 2. Format: gặp "Bước X:" thì xuống dòng
            steps_text = re.sub(r'(Bước\s+\d+:)', r'\n\1', steps_text).strip()

            print(f"\nCÁCH LÀM:")
            print(steps_text)

        # Notes
        notes = parse_list_field(row['note'])
        if notes:
            print(f"\nLƯU Ý:")
            for i, note in enumerate(notes, 1):
                print(f"   • {note}")

        print()

In [28]:
# Test Late Fusion
test_queries = [
    "Món ăn có thịt bò nấu nhanh",
]

# Run Late Fusion tests
for query in test_queries:
    results = search_recipes_late_fusion(
        query=query,
        model=model,
        recipes_embeddings_list=recipes_embeddings_list,
        all_recipes_df=all_recipes_df,
        top_k=5
    )

    # Display results
    display_results(results, query)

Query: 'Món ăn có thịt bò nấu nhanh'
Top 5 Results:

1. [0.6043] Bò hầm cà rốt

THÔNG TIN CƠ BẢN:
   • Loại món: Món chính
   • Thời gian nấu: 30phút
   • Số người ăn: 2
   • Link: https://vncooking.com/cong-thuc/bo-ham-ca-rot-14

ĐIỂM SIMILARITY:
   • Trung bình (AVG): 0.6043
   • Cao nhất (MAX): 0.7382
   • Thấp nhất (MIN): 0.4551
   • Số câu đánh giá: 4

MÔ TẢ:
   Bò luôn là món thịt mà đa phần các gia đình Việt rất ưa chuộng, bò chứa hàm lượng dinh dưỡng siêu cao cộng với cà rốt nữa làm tăng thêm phần dinh dưỡng của món ăn. Cùng chuẩn bị nguyên liệu thực hiện món Bò hầm cà rốt này nhé.

NGUYÊN LIỆU (7 món):
   1. Thịt bò 150 gram
   2. Cà rốt 2 củ
   3. Nước mắm 1 muỗng cafe
   4. Gừng 1 củ
   5. Muối 1 muỗng
   6. Dầu ăn 2 muỗng
   7. Sả 1 cây

CÁCH LÀM:
Bước 1: Nguyên liệu rửa sạch. Cà rốt cạo vỏ thái thành những hình vuông nhỏ, vừa ăn. Thịt bò cũng vậy, thái thành từng miếng hình vuống nhỏ thôi cho không bị day nha. Các nguyên liệu khác bỏ vỏ đun trên 1 nồi nước nhỏ, chờ khi nướ

### 5. Interactive Search

In [29]:
# Interactive search với Late Fusion
print("Enter your query (type 'quit' to exit):\n")

while True:
    query = input("Query: ").strip()

    if query.lower() in ['quit', 'exit', 'q']:
        break

    if not query:
        continue

    results = search_recipes_late_fusion(
        query=query,
        model=model,
        recipes_embeddings_list=recipes_embeddings_list,
        all_recipes_df=all_recipes_df,
        top_k=10
    )

    # Display results
    display_results(results, query)

Enter your query (type 'quit' to exit):



In [30]:
results.head()

,recipe_idx,avg_similarity,max_similarity,min_similarity,num_sentences,title,type_of_food,cook_time,num_of_people,ingredients,step,note,description,link
0,1268,0.604319,0.738170,0.455103,4,Bò hầm cà rốt,Món chính,30phút,2,"['Thịt bò 150 gram', 'Cà rốt 2 củ', 'Nước mắm ...",['Bước 1: Nguyên liệu rửa sạch. Cà rốt cạo vỏ ...,[],Bò luôn là món thịt mà đa phần các gia đình Vi...,https://vncooking.com/cong-thuc/bo-ham-ca-rot-14
1,4316,0.595165,0.715963,0.341206,7,Phở bò bằng gói gia vị sẵn thơm lừng chuẩn vị ...,Món nước,55 phút,4 người,"['500 gr Phở tươi', '300 gr Thịt bò', '500 gr ...",['Bước 1: Sơ chế nguyên liệu: Để khử mùi hôi c...,['Xem chi tiết: Cách chọn mua thịt bò tươi ngo...,"Phở bò là một món ăn thơm ngon, đặc trưng tron...",https://www.dienmayxanh.com/vao-bep/cach-nau-p...
2,5630,0.592639,0.665365,0.504305,5,Thịt bò xào cần tây đơn giản thơm ngon dậy mùi...,Món xào,10 phút,2 người,"['300 gr Thịt bò', '1 bó Cần tây (khoảng 100gr...","['Bước 1: Sơ chế nguyên liệu: Đầu tiên, thịt b...",['Xem chi tiết: Cách chọn mua thịt bò tươi ngo...,Thịt bò xào là một trong những món xào được nh...,https://www.dienmayxanh.com/vao-bep/cach-lam-t...
3,341,0.588460,0.740761,0.395903,5,Nui xào thịt bò,Món ngon hàng ngày,NaN,NaN,"['250 g thịt bò xay nhuyễn', '150 g nui', '1 c...","['Bước 1: Trộn thịt bò với hành tây, tỏi, cà r...",[],"Hình dáng, màu sắc của những sợi nui quện lẫn ...",https://vnexpress.net/nui-xao-thit-bo-4311129....
4,4200,0.587533,0.657737,0.514986,6,"Miến thịt bò ngon ngọt, hấp dẫn đổi vị cho bữa...",Món nước,40 phút,4 người ăn,"['250 gr Huyết bò', '350 gr Thịt bò', '100 gr ...",['Bước 1: Sơ chế huyết bò và thịt bò: Huyết bò...,"['Xem chi tiết: Cách phân loại miến, chọn miến...",Miến thịt bò - một trong những món nước mang h...,https://www.dienmayxanh.com/vao-bep/cach-nau-m...


### 6. Combine with state.json

In [31]:
import json
import google.generativeai as genai
import os
from dotenv import load_dotenv
from pathlib import Path

# Load API key
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)

MODEL = "models/gemini-2.5-flash-lite"  # Fast and efficient model
STATE_FILE = "state.json"

#### 6.1 Load state.json function

In [44]:
def load_state_json(filepath=STATE_FILE):
    """Load dialogue state from JSON file"""
    try:
        if Path(filepath).exists():
            with open(filepath, 'r', encoding='utf-8') as f:
                state = json.load(f)
            return state
        else:
            print(f"[Error]: {filepath} not found")
            return None
    except json.JSONDecodeError:
        print(f"[Error]: Invalid JSON in {filepath}")
        return None

In [45]:
state = load_state_json()
state

{'hard_constraints': {'type_of_food': ['món kho'],
  'ingredients': ['thịt heo']},
 'soft_constraints': {'cook_time': ['trung bình'],
  'num_of_people': ['4'],
  'calories': ['none'],
  'algeric': ['none']},
 'recommended_items': [],
 'accepted_items': [],
 'rejected_items': []}

#### 6.2 Generate_query_rule_based (if Gemini out of quota)

In [46]:
def generate_query_rule_based(state):
    """Rule-based query generation (fallback)"""
    parts = []
    
    # Hard constraints
    if "hard_constraints" in state:
        if state["hard_constraints"].get("type_of_food"):
            parts.append(state["hard_constraints"]["type_of_food"][0])
        if state["hard_constraints"].get("ingredients"):
            ings = ', '.join(state["hard_constraints"]["ingredients"])
            parts.append(f"có {ings}")
    
    # Soft constraints
    if "soft_constraints" in state:
        if state["soft_constraints"].get("cook_time"):
            ct = state["soft_constraints"]["cook_time"]
            if ct and ct[0] != "none":
                parts.append(f"nấu {ct[0]}")
        if state["soft_constraints"].get("num_of_people"):
            np = state["soft_constraints"]["num_of_people"]
            if np and np[0] != "none":
                parts.append(f"cho {np[0]} người")
    
    query = "Món ăn " + ", ".join(parts) if parts else "Món ăn"
    return query

In [47]:
test = generate_query_rule_based(state)
test

'Món ăn món kho, có thịt heo, nấu trung bình, cho 4 người'

#### 6.3 Generate_query_from_state (use gemini)

In [ ]:
def generate_query_from_state(state, use_gemini=True):
    """
    Generate search query from state.json constraints
    
    Args:
        state: Dictionary from state.json (with hard_constraints, soft_constraints)
        use_gemini: If True, use Gemini API; else use rule-based generation
    
    Returns:
        Generated query string
    """
    if use_gemini:
        # Prepare constraints text
        constraints_text = "Constraints từ state.json:\n"
        
        # Hard constraints
        if "hard_constraints" in state:
            constraints_text += "\nHard Constraints:\n"
            for key, values in state["hard_constraints"].items():
                if values:
                    constraints_text += f"  - {key}: {', '.join(map(str, values))}\n"
        
        # Soft constraints
        if "soft_constraints" in state:
            constraints_text += "\nSoft Constraints:\n"
            for key, values in state["soft_constraints"].items():
                if values and values != ["none"]:
                    constraints_text += f"  - {key}: {', '.join(map(str, values))}\n"
        
        # Use Gemini to generate natural query
        prompt = f"""Bạn là trợ lý tìm kiếm món ăn. Dựa vào thông tin constraints sau, hãy tạo 1 câu tìm kiếm món ăn TỰ NHIÊN, NGẮN GỌN bằng tiếng Việt:

                    {constraints_text}

                    YÊU CẦU:
                    - Câu phải chứa TẤT CẢ thông tin có trong constraints (trừ các giá trị "none")
                    - Câu phải tự nhiên, dễ hiểu, như cách người Việt nói chuyện
                    - Không dài dòng, chỉ 1-2 câu ngắn gọn
                    - Không cần nhắc lại từ "tìm kiếm", chỉ cần mô tả món ăn
                    - Ưu tiên hard constraints hơn soft constraints

                    VÍ DỤ:
                    - Input: type_of_food: ["món chính"], ingredients: ["thịt bò"], cook_time: ["nhanh"]
                    Output: "Món chính có thịt bò nấu nhanh"

                    - Input: type_of_food: ["món Tết"], ingredients: ["thịt lợn"], num_of_people: ["4"]
                    Output: "Món Tết có thịt lợn cho 4 người"

                    Chỉ trả về câu tìm kiếm, KHÔNG giải thích thêm:"""

        try:
            response = genai.GenerativeModel(MODEL).generate_content(prompt)
            query = response.text.strip()
            # Remove quotes if present
            query = query.strip('"').strip("'")
            return query
        except Exception as e:
            print(f"❌ Error generating query with Gemini: {e}")
            print("⚠️ Falling back to rule-based generation...")
            return generate_query_rule_based(state)
    else:
        return generate_query_rule_based(state)

In [37]:
test2 = generate_query_from_state(state)
test2

'Món chính thịt kho nấu trung bình cho 4 người.'

#### 6.4 Search

In [49]:
def search_from_state_json(state_filepath=STATE_FILE, use_gemini=True, top_k=10):
    """
    Main function: Load state.json → Generate query → Search → Display
    
    Args:
        state_filepath: Path to state.json file
        use_gemini: Use Gemini API (True) or rule-based generation (False)
        top_k: Number of results to return
    
    Returns:
        DataFrame with search results
    """
    # STEP 1: Load state.json
    print("\STEP 1: Loading state.json...")
    print("-"*100)
    state = load_state_json(state_filepath)
    
    if state is None:
        return None
    
    # STEP 2: Generate query
    print("STEP 2: Generating query from constraints...")
    print("-"*100)
    
    query = generate_query_from_state(state, use_gemini=use_gemini)
    print(f"Generated Query: '{query}'")
    
    # STEP 3: Search recipes using Late Fusion
    print("STEP 3: Searching recipes with Late Fusion...")
    print("-"*100)
    
    results = search_recipes_late_fusion(
        query=query,
        model=model,
        recipes_embeddings_list=recipes_embeddings_list,
        all_recipes_df=all_recipes_df,
        top_k=top_k
    )
    
    print(f"\nFound {len(results)} recipes")
    
    # STEP 4: Display results
    print("\n" + "="*100)
    print("STEP 4: RESULTS")
    print("="*100)
    display_results(results, query)
    
    return results

<>:14: SyntaxWarning: invalid escape sequence '\S'
<>:14: SyntaxWarning: invalid escape sequence '\S'
C:\Users\Admin\AppData\Local\Temp\ipykernel_6372\217843517.py:14: SyntaxWarning: invalid escape sequence '\S'
  print("\STEP 1: Loading state.json...")


In [50]:
# Run search from state.json with Gemini
results = search_from_state_json(
    state_filepath="state.json",
    use_gemini=True,  # Set to False for rule-based generation (no API needed)
    top_k=10
)

\STEP 1: Loading state.json...
----------------------------------------------------------------------------------------------------
STEP 2: Generating query from constraints...
----------------------------------------------------------------------------------------------------
Generated Query: 'Món kho thịt heo cho 4 người nấu trung bình'
STEP 3: Searching recipes with Late Fusion...
----------------------------------------------------------------------------------------------------

Found 10 recipes

STEP 4: RESULTS
Query: 'Món kho thịt heo cho 4 người nấu trung bình'
Top 10 Results:

1. [0.6221] Bún sườn heo hầm

THÔNG TIN CƠ BẢN:
   • Loại món: Món chính
   • Thời gian nấu: 45phút
   • Số người ăn: 4
   • Link: https://vncooking.com/cong-thuc/bun-suon-heo-ham-126

ĐIỂM SIMILARITY:
   • Trung bình (AVG): 0.6221
   • Cao nhất (MAX): 0.7703
   • Thấp nhất (MIN): 0.5503
   • Số câu đánh giá: 4

MÔ TẢ:
   Bún nấu cùng sườn heo hầm đậm đà sẽ là bữa ăn dành cho gia đình bạn tối nay?
Hãy cù